# Tutorial: ROMA Agent Architecture and Execution Model

Audience:
- Agent architects evaluating decomposition/orchestration protocols in MAS systems.
- AI engineers implementing production-grade task execution loops.
- Data scientists who need to understand where planning quality impacts final outcomes.

Prerequisites:
- Comfortable with Python and async execution.
- Familiar with graph-based task decomposition.
- ROMA repository already installed in a Python 3.12 environment.

Learning goals:
- Understand ROMA's 5-agent architecture and responsibility boundaries.
- Understand how planning states map to task lifecycle states in code.
- Inspect the exact orchestration path implemented in `RecursiveSolver` and `ModuleRuntime`.


## Outline

1. Environment + source binding.
2. Architecture map and role responsibilities.
3. Signature contract and type system.
4. Agent registry/factory initialization path.
5. State-machine orchestration and recursion logic.
6. Role-based reading strategy (architect / data scientist / AI engineer).


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "roma_dspy").exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / "src" / "roma_dspy").exists():
            REPO_ROOT = candidate
            break

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH added:", SRC_PATH)


REPO_ROOT: /Users/admin/TuanDung/repos/ROMA
SRC_PATH added: /Users/admin/TuanDung/repos/ROMA/src


## Architecture At A Glance (Code-Accurate)

ROMA runtime centers on a **TaskDAG + recursive state machine**:

```text
TaskNode(PENDING)
  -> Atomizer decides NodeType (PLAN|EXECUTE)
  -> if EXECUTE: Executor -> COMPLETED
  -> if PLAN:
       Planner -> subtasks + dependencies_graph
       create subgraph (index deps -> task_id deps)
       solve ready subtasks in parallel
       Aggregator synthesizes child outputs
       parent -> COMPLETED
```

Core files to inspect:
- `src/roma_dspy/core/engine/solve.py`
- `src/roma_dspy/core/engine/runtime.py`
- `src/roma_dspy/core/engine/dag.py`
- `src/roma_dspy/core/signatures/signatures.py`


In [2]:
from pathlib import Path

key_files = [
    "src/roma_dspy/core/engine/solve.py",
    "src/roma_dspy/core/engine/runtime.py",
    "src/roma_dspy/core/engine/dag.py",
    "src/roma_dspy/core/signatures/signatures.py",
    "src/roma_dspy/core/signatures/base_models/subtask.py",
]

for rel in key_files:
    p = REPO_ROOT / rel
    print(f"{rel}:", "OK" if p.exists() else "MISSING")


src/roma_dspy/core/engine/solve.py: OK
src/roma_dspy/core/engine/runtime.py: OK
src/roma_dspy/core/engine/dag.py: OK
src/roma_dspy/core/signatures/signatures.py: OK
src/roma_dspy/core/signatures/base_models/subtask.py: OK


## Signature Contract And Type System

Planning/orchestration behavior is constrained by:
- `TaskType` (MECE classes: RETRIEVE/WRITE/THINK/CODE_INTERPRET/IMAGE_GENERATION)
- `NodeType` (PLAN or EXECUTE from atomizer decision)
- `TaskStatus` (state machine progression)
- `PlannerSignature` and `SubTask` schema

This is the contract that keeps planning output machine-consumable.


In [3]:
import inspect
from roma_dspy.types import TaskType, NodeType, TaskStatus
from roma_dspy.core.signatures.signatures import (
    AtomizerSignature,
    PlannerSignature,
    ExecutorSignature,
    AggregatorSignature,
    VerifierSignature,
)
from roma_dspy.core.signatures.base_models.subtask import SubTask

print("TaskType:", [t.value for t in TaskType])
print("NodeType:", [n.value for n in NodeType])
print("TaskStatus:", [s.value for s in TaskStatus])
print()
print("SubTask JSON schema keys:", list(SubTask.model_json_schema().keys()))

print()
print("PlannerSignature source excerpt:")
print(chr(10).join(inspect.getsource(PlannerSignature).splitlines()[:35]))


2026-02-24 13:46:52.644 | INFO     | roma_dspy.core.predictors.code_act_patch:apply_code_act_patch:96 - Applied CodeAct patch to inject typing imports into interpreter


TaskType: ['RETRIEVE', 'WRITE', 'THINK', 'CODE_INTERPRET', 'IMAGE_GENERATION']
NodeType: ['PLAN', 'EXECUTE']
TaskStatus: ['PENDING', 'ATOMIZING', 'PLANNING', 'PLAN_DONE', 'READY', 'EXECUTING', 'AGGREGATING', 'COMPLETED', 'FAILED', 'NEEDS_REPLAN']

SubTask JSON schema keys: ['$defs', 'description', 'properties', 'required', 'title', 'type']

PlannerSignature source excerpt:
class PlannerSignature(dspy.Signature):
    """
    Planner decomposition result.

    Contains the breakdown of a complex task into executable subtasks.
    """

    goal: str = dspy.InputField(
        description="Task that needs to be decomposed into subtasks through planner"
    )
    context: Optional[str] = dspy.InputField(
        default=None, description="Execution context (XML)"
    )
    subtasks: List[SubTask] = dspy.OutputField(
        description="List of generated subtasks from planner"
    )
    dependencies_graph: Optional[Dict[str, List[str]]] = dspy.OutputField(
        default=None,
        desc

## Agent Registry + Factory Path

Initialization flow:
1. `ConfigManager` resolves YAML + env into `ROMAConfig`.
2. `AgentFactory` resolves signature/instructions and creates module instances.
3. `AgentRegistry` stores `(AgentType, TaskType?) -> module` with fallback to default.

This design lets ROMA support task-type-specific agents without changing orchestration code.


In [4]:
from roma_dspy.config.manager import ConfigManager
from roma_dspy.core.factory.agent_factory import AgentFactory
from roma_dspy.core.registry.agent_registry import AgentRegistry

cfg_manager = ConfigManager(config_dir=REPO_ROOT / "config")
cfg_path = REPO_ROOT / "config" / "local" / "openai_config.yaml"
if not cfg_path.exists():
    cfg_path = REPO_ROOT / "config" / "defaults" / "config.yaml"

config = cfg_manager.load_config(config_path=str(cfg_path))
print("Loaded config:", cfg_path)

registry = AgentRegistry()
registry.initialize_from_config(config, factory=AgentFactory())
print("Registry stats:", registry.get_stats())

for agent_type, task_type, module in registry.iter_agents():
    model_cfg = module.get_model_config()
    print({
        "agent_type": str(agent_type),
        "task_type": str(task_type) if task_type else "default",
        "model": model_cfg.get("model"),
        "temperature": model_cfg.get("kwargs", {}).get("temperature"),
    })


2026-02-24 13:46:52.749 | DEBUG    | roma_dspy.config.manager:load_config:74 - Loading config: path=/Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml, profile=None, overrides=None, env_prefix=ROMA_
2026-02-24 13:46:52.750 | DEBUG    | roma_dspy.config.manager:load_config:83 - Initialized empty base config (defaults applied in validation)
2026-02-24 13:46:52.765 | DEBUG    | roma_dspy.config.manager:_load_yaml:149 - Loaded and cached config from /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
2026-02-24 13:46:52.766 | DEBUG    | roma_dspy.config.manager:load_config:93 - Merged YAML config from /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
2026-02-24 13:46:52.776 | DEBUG    | roma_dspy.config.manager:load_config:123 - Resolved interpolations
2026-02-24 13:46:52.777 | INFO     | roma_dspy.config.manager:load_config:130 - Configuration loaded and validated successfully
2026-02-24 13:46:52.779 | DEBUG    | roma_dspy.core.factory.agent_facto

Loaded config: /Users/admin/TuanDung/repos/ROMA/config/local/openai_config.yaml
Registry stats: {'registrations': 5, 'lookups': 0, 'fallbacks': 0, 'cache_hits': 0, 'total_agents': 5, 'task_specific': 0, 'defaults': 5}
{'agent_type': 'atomizer', 'task_type': 'default', 'model': 'openai/gpt-4o-mini', 'temperature': 0.1}
{'agent_type': 'planner', 'task_type': 'default', 'model': 'openai/gpt-4o-mini', 'temperature': 0.3}
{'agent_type': 'executor', 'task_type': 'default', 'model': 'openai/gpt-4o', 'temperature': 0.7}
{'agent_type': 'aggregator', 'task_type': 'default', 'model': 'openai/gpt-4o-mini', 'temperature': 0.3}
{'agent_type': 'verifier', 'task_type': 'default', 'model': 'openai/gpt-4o-mini', 'temperature': 0.1}


## State Machine Logic (From Runtime)

The key transition logic lives in `RecursiveSolver._async_execute_state_machine`.

Pseudocode:

```text
if should_force_execute():
    force_execute_async
elif status == PENDING:
    atomize_async
if status == ATOMIZING:
    transition_from_atomizing
if status == PLANNING:
    plan_async (+ optional checkpoint)
if status == EXECUTING:
    execute_async
elif status == PLAN_DONE:
    process_subgraph_async -> aggregate_async
```

This makes planning and execution deterministic from task state.


In [5]:
import inspect
from roma_dspy.core.engine.solve import RecursiveSolver
from roma_dspy.core.engine.runtime import ModuleRuntime

sm_src = inspect.getsource(RecursiveSolver._async_execute_state_machine).splitlines()
rt_plan_src = inspect.getsource(ModuleRuntime.plan_async).splitlines()

print("_async_execute_state_machine (first 60 lines):")
print(chr(10).join(sm_src[:60]))

print()
print("ModuleRuntime.plan_async (first 45 lines):")
print(chr(10).join(rt_plan_src[:45]))


_async_execute_state_machine (first 60 lines):
    async def _async_execute_state_machine(
        self, task: TaskNode, dag: TaskDAG, checkpoint_id: Optional[str] = None
    ) -> TaskNode:
        """Execute asynchronous state machine for task processing."""
        # Check for forced execution at max depth
        if task.should_force_execute():
            logger.debug(f"Force executing task at max depth: {task.depth}")
            return await self.runtime.force_execute_async(task, dag)

        # Process based on current state
        if task.status == TaskStatus.PENDING:
            logger.debug(f"Async atomizing task: {task.goal[:50]}...")
            task = await self.runtime.atomize_async(task, dag)

        if task.status == TaskStatus.ATOMIZING:
            task = self.runtime.transition_from_atomizing(task, dag)

        if task.status == TaskStatus.PLANNING:
            logger.debug(f"Async planning task: {task.goal[:50]}...")
            task = await self.runtime.plan_asy

## Role-Based Reading Strategy

For Agent Architect:
- Focus on `TaskStatus` transitions and checkpoint boundaries.
- Evaluate whether `PLAN_DONE -> process_subgraph_async -> aggregate_async` matches your policy requirements.

For Data Scientist:
- Focus on planner output contract (`subtasks`, `dependencies_graph`) and result synthesis quality in aggregator.
- Measure decomposition quality (fan-out, dependency density, plan depth proxy).

For AI Engineer:
- Focus on `AgentFactory`, runtime context/tooling, and failure/recovery paths.
- Verify provider config, timeouts, and checkpoint behavior under long runs.


## Exercise

1. Replace one planner model in config and re-run registry inspection.
2. Compare resulting planner behavior in Notebook 2 (plan shape and dependency graph).
3. Validate orchestration behavior in Notebook 3 using recursive vs event-driven execution.
